In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge

In [2]:
df_reviews = pd.read_csv('All_Beauty.csv')[['user_id', 'parent_asin', 'rating']].dropna()
df_reviews.columns = ['userId', 'productId', 'score']

df_meta = pd.read_csv('meta_All_Beauty.csv')[['parent_asin', 'title']].dropna()
df_meta.columns = ['productId', 'title']

df_reviews = df_reviews[df_reviews['productId'].isin(df_meta['productId'])]

# Lọc người dùng: giữ các người dùng đã đánh giá ít nhất 5 sản phẩm
user_counts = df_reviews['userId'].value_counts()
df_reviews = df_reviews[df_reviews['userId'].isin(user_counts[user_counts >= 5].index)]

print(len(df_reviews))
print(df_reviews['userId'].nunique())
df_reviews['productId'].nunique()

14983
1620


6981

Số lượng review còn lại: 14983
Số users: 1620 | Số products: 6981


In [3]:
# Loại bỏ các sản phẩm trùng lặp trong metadata
df_meta = df_meta.drop_duplicates('productId')

# Trích xuất đặc trưng tiêu đề sản phẩm bằng TF-IDF
# Lấy 2000 từ khóa đặc trưng nhất của hệ thống
# Loại bỏ các từ phụ phổ biến trong tiếng Anh
tfidf = TfidfVectorizer(max_features=2000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_meta['title'])

# Lưu vector của từng sản phẩm vào một từ điển (dict) để tra cứu nhanh theo ID sản phẩm
product2vector = {row['productId']: tfidf_matrix[i] for i, row in df_meta.reset_index(drop=True).iterrows()}
print(f"Đã tạo vector đặc trưng TF-IDF cho {len(product2vector)} sản phẩm.")

Đã tạo vector đặc trưng TF-IDF cho 112578 sản phẩm.


In [4]:
# Chia tập dữ liệu thành 80% Train và 20% Test
train_df, test_df = train_test_split(df_reviews, test_size=0.2, random_state=42)
print(f"Kích thước tập Train: {len(train_df)} | Tập Test: {len(test_df)}")

Kích thước tập Train: 11986 | Tập Test: 2997


In [5]:
# Điểm đánh giá trung bình của từng user trong tập Train (để làm mốc rating mặc định)
user_mean = train_df.groupby('userId')['score'].mean().to_dict()
global_mean = train_df['score'].mean()

# Huấn luyện mô hình Ridge Regression cho từng người dùng
user_profiles_W = {}
user_profiles_b = {}

print("Đang huấn luyện mô hình Ridge Regression cho từng người dùng...")
for user_id, group in train_df.groupby('userId'):
    pids = group['productId'].tolist()
    ratings = group['score'].tolist()
    
    X_user = []
    y_user = []
    for pid, rating in zip(pids, ratings):
        if pid in product2vector:
            X_user.append(product2vector[pid].toarray().flatten())
            y_user.append(rating)
            
    if len(X_user) > 0:
        X_user = np.array(X_user)
        y_user = np.array(y_user)
        
        # Huấn luyện Ridge Regression với alpha = 0.01
        clf = Ridge(alpha=0.01, fit_intercept=True)
        clf.fit(X_user, y_user)
        
        user_profiles_W[user_id] = clf.coef_
        user_profiles_b[user_id] = clf.intercept_

print(f"Đã huấn luyện mô hình cho {len(user_profiles_W)} người dùng.")

Đã học mô hình sở thích cho 1552 người dùng.


In [6]:
# Hàm đánh giá RMSE cho tập dữ liệu
def evaluate_rmse(df, user_profiles_W, user_profiles_b, product2vector, user_mean, global_mean):
    sq_errors = []
    for _, row in df.iterrows():
        u, p, actual = row['userId'], row['productId'], row['score']
        
        # Chỉ dự đoán nếu người dùng đã có hồ sơ và sản phẩm có vector đặc trưng
        if u in user_profiles_W and p in product2vector:
            W_u = user_profiles_W[u]
            b_u = user_profiles_b[u]
            x_p = product2vector[p].toarray().flatten()
            
            pred = np.dot(x_p, W_u) + b_u
            pred = np.clip(pred, 1.0, 5.0) # Giới hạn điểm rating từ 1.0 đến 5.0 sao
            sq_errors.append((actual - pred) ** 2)
        else:
            # Dự đoán mặc định bằng điểm trung bình của user đó, hoặc điểm trung bình toàn bộ tập train
            pred = user_mean.get(u, global_mean)
            sq_errors.append((actual - pred) ** 2)
            
    return np.sqrt(np.mean(sq_errors))

print("Đang đánh giá sai số RMSE của mô hình Ridge Regression...")
rmse_train = evaluate_rmse(train_df, user_profiles_W, user_profiles_b, product2vector, user_mean, global_mean)
rmse_test = evaluate_rmse(test_df, user_profiles_W, user_profiles_b, product2vector, user_mean, global_mean)

print(f"==> Sai số RMSE trên tập Train: {rmse_train:.4f}")
print(f"==> Sai số RMSE trên tập Test: {rmse_test:.4f}")

Đang dự đoán điểm và đánh giá mô hình trên tập Test...
==> Sai số RMSE của mô hình Content-Based đơn giản: 1.0575


==> Sai số RMSE của mô hình Content-Based đơn giản: 1.0758


In [7]:
def recommend(user_id, top_n=5):
    """
    Hàm gợi ý danh sách top_n sản phẩm chưa mua có điểm dự đoán cao nhất cho người dùng.
    """
    if user_id not in user_profiles_W:
        return "Người dùng mới hoặc chưa có dữ liệu sở thích."
    
    W_u = user_profiles_W[user_id]
    b_u = user_profiles_b[user_id]
    
    # Lấy danh sách tất cả các sản phẩm
    all_pids = list(product2vector.keys())
    
    # Tính điểm dự đoán cho tất cả sản phẩm
    # Sử dụng nhân ma trận tfidf_matrix (sparse) với vector trọng số W_u để tối ưu hiệu năng
    scores = tfidf_matrix.dot(W_u) + b_u
    scores = np.clip(scores, 1.0, 5.0)
    
    # Tìm các sản phẩm người dùng đã xem ở tập Train để lọc bỏ
    watched = set(train_df[train_df['userId'] == user_id]['productId'])
    
    recommendations = []
    for idx, pid in enumerate(all_pids):
        if pid not in watched:
            recommendations.append((pid, scores[idx]))
            
    # Sắp xếp các sản phẩm theo điểm dự đoán giảm dần
    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:top_n]

# Chạy thử nghiệm gợi ý cho một người dùng ngẫu nhiên
sample_user = list(user_profiles_W.keys())[0]
print(f"Gợi ý sản phẩm cho người dùng: '{sample_user}'")
recs = recommend(sample_user, top_n=3)
for i, (pid, score) in enumerate(recs):
    title_matches = df_meta[df_meta['productId'] == pid]['title']
    title = title_matches.iloc[0] if len(title_matches) > 0 else 'Không tìm thấy tên'
    print(f"  {i+1}. Mã: {pid} | Điểm dự đoán: {score:.4f} | Tên: {title[:70]}...")

Gợi ý sản phẩm cho người dùng: 'AE23ZBUF2YVBQPH2NN6F5XSA3QYQ'
  1. Mã: B08CVTSBKT | Độ tương thích: 0.4087 | Tên: Disney Frozen 2 - Townley Girl Mega Hair Foldable Make-up Gift Set for...
  2. Mã: B08BFBYNM1 | Độ tương thích: 0.4080 | Tên: Facial Kit For Women - Includes Facial Mask, Facial Makeup Wipes, Nose...
  3. Mã: B08BFPN1PY | Độ tương thích: 0.4047 | Tên: Facial Kit For Women - Includes Facial Mask, Facial Makeup Wipes, Nose...


  1. Mã: B01LZA8QP4 | Độ tương thích: 0.4946 | Tên: Bodycology Sweet Love Gift Set With Warm and Cozy Socks...
  2. Mã: B003O9RM9Q | Độ tương thích: 0.4946 | Tên: Bvlgari Aqva By Bvlgari Gift Set...
  3. Mã: B01AUYRXSW | Độ tương thích: 0.4946 | Tên: Vaseline Cocoa Radiant Holiday Gift Set...
